### Group info

ID: G5_4

Members:
- Vinicius Millan Santa Catharina
- Zaur Kambiev
- Stig Espen Paulsberg

### Imports

In [18]:
import numpy as np
import pandas as pd
from numpy.typing import ArrayLike, NDArray
from pathlib import Path
import re

### Task 1

In [19]:
def file_reader(file_path) -> tuple[str, str]:
    '''Reads files and returns tuple of text content and encoding.
    
    Read through log files, first as UTF-8 then ISO-8859-1 if it fails.'''
    try:
        # Try reading as UTF-8
        with open(
            file=file_path,
            mode='r',
            encoding='utf-8'
        ) as file:
            file_text_content = file.read()
            encoding_used = file.encoding
    except UnicodeDecodeError:
        # If reading UTF-8 fails, read as ISO-8859-1 instead
        with open(
            file=file_path,
            mode='r',
            encoding='cp1252' # equivalent to iso-8859-1
        ) as file:
            file_text_content = file.read()
            encoding_used = file.encoding
    return file_text_content, encoding_used

In [20]:
def log_parser(file_paths):
    '''Searches through file_paths looking for pattern matches.
    
    Returns a list of dictionaries representing valid transaction records.
    '''
    records = []
    
    # Patterns to match
    log_patterns = re.compile(
        r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s"
        r"\[(?P<register_id>[\w_]+)\]\s+"
        r"item=(?P<item>[A-Za-z]+)\s+"
        r"price=(?P<price>\d+\.\d+)\s+"
        r"(?P<currency>[A-Z]{3})\s+"
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
    )

    # Loop over every log_file
    for log_file in file_paths:
        # Loop over every line in text content of each log_file
        log_file_content = file_reader(log_file)[0].splitlines()
        for line in log_file_content:
            match = log_patterns.search(line.strip())
            if match:
                data = match.groupdict()
                record = {
                    "timestamp":    data["timestamp"],
                    "register_id":  data["register_id"],
                    "item":         data["item"],
                    "price":  float(data["price"]),
                    "currency":     data["currency"],
                    "status":       data["status"],
                }
                records.append(record)
    return records

In [21]:
data_log_files_path = Path('../data').resolve().rglob('*.log')

transaction_records = log_parser(data_log_files_path)

### Task 2

### Task 3

Define a function for handling price outliers.

In [22]:
def handle_outliers(prices: ArrayLike) -> NDArray[np.float64]:
    """Replace price outliers with the median of valid prices.

    Outliers are defined as prices more than three standard deviations
    away from the mean.

    Parameters
    ----------
    prices : array_like
        Prices convertible to a 1D NumPy array of floats.

    Returns
    -------
    ndarray
        A 1D NumPy array of prices with outliers replaced by the
        median of valid prices.
    """

    prices = np.asarray(prices, dtype=np.float64)

    if prices.ndim != 1:
        raise ValueError("Prices must be 1D.")

    mean = prices.mean()
    std = prices.std()

    valid_mask = abs(prices - mean) <= 3 * std

    median = np.median(prices[valid_mask])

    prices[~valid_mask] = median

    return prices

Define a function for calculating the refund ratio of items.

In [23]:
EPS = 1e-6


def compute_refund_ratio(
        sales_count: NDArray,
        refund_count: NDArray
) -> NDArray:
    """Compute the refund-to-sale ratio for each item.

    A small positive constant (EPS) is added to the number of sales
    to prevent division by zero.

    Parameters
    ----------
    sales_count : ndarray
        Number of sales for each item.
    refund_count : ndarray
        Number of refunds for each item.

    Returns
    -------
    ndarray
        An array containing element-wise refund-to-sale ratios.
    """

    return refund_count / (sales_count + EPS)

### Task 4

Define a function that converts a list of transactions into a `pandas.DataFrame`.

In [24]:
def list_to_df(transactions: list[dict]) -> pd.DataFrame:
    """Convert a list of transactions into a pandas DataFrame.

    Convert the timestamp column to datetime, set it as the DataFrame index,
    and sort the transactions chronologically.

    Parameters
    ----------
    transactions : list[dict]
        A list of dictionaries containing a `timestamp` field
        in YYYY-MM-DD HH:MM:SS format.

    Returns
    -------
    pd.DataFrame
        A DataFrame indexed by timestamp, sorted in chronological order.
    """

    transactions = pd.DataFrame(transactions)

    transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

    transactions = transactions.set_index("timestamp").sort_index()

    return transactions

In [29]:
df = list_to_df(transaction_records)
df.head(20)

,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:21,POS_01,Cappuccino,4.30,NOK,CANCELLED
2026-10-15 07:01:00,POS_01,Sandwich,7.50,EUR,SUCCESS
2026-10-15 07:01:14,POS_02,Espresso,2.67,EUR,SUCCESS
2026-10-15 07:01:23,POS_01,Croissant,3.51,EUR,SUCCESS
2026-10-15 07:01:28,POS_02,Croissant,3.96,EUR,SUCCESS
2026-10-15 07:01:50,POS_03,Sandwich,7.15,EUR,SUCCESS
2026-10-15 07:01:58,POS_02,Latte,4.37,EUR,REFUND
2026-10-15 07:02:03,POS_01,Muffin,3.27,NOK,SUCCESS
2026-10-15 07:02:23,POS_03,Croissant,3.56,USD,SUCCESS


Define a function that apply currency correction all NOK currency transactions.

In [34]:
def correct_nok_prices(transactions: pd.DataFrame) -> pd.DataFrame:
    """Correct the price glitch in NOK transactions."""
    
    result = transactions.copy()
    result.loc[result['currency'] == 'NOK', 'price']*=10

    return result

df2 = correct_nok_prices(df)
df2.head(20)

,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:21,POS_01,Cappuccino,43.00,NOK,CANCELLED
2026-10-15 07:01:00,POS_01,Sandwich,7.50,EUR,SUCCESS
2026-10-15 07:01:14,POS_02,Espresso,2.67,EUR,SUCCESS
2026-10-15 07:01:23,POS_01,Croissant,3.51,EUR,SUCCESS
2026-10-15 07:01:28,POS_02,Croissant,3.96,EUR,SUCCESS
2026-10-15 07:01:50,POS_03,Sandwich,7.15,EUR,SUCCESS
2026-10-15 07:01:58,POS_02,Latte,4.37,EUR,REFUND
2026-10-15 07:02:03,POS_01,Muffin,32.70,NOK,SUCCESS
2026-10-15 07:02:23,POS_03,Croissant,3.56,USD,SUCCESS


Define a function that add a columns with the prices in nok.

In [35]:
def add_nok_prices(transactions: pd.DataFrame) -> pd.DataFrame:
    """Add a column with transaction prices converted to NOK."""

    conversion_rates = {
        "NOK": 1.0,
        "EUR": 10.4,
        "USD": 9.3
    }

    result = transactions.copy()
    result['nok_price'] = (result['currency'].map(conversion_rates)*result['price']).round(2)

    return result

add_nok_prices(df2)


,register_id,item,price,currency,status,nok_price
timestamp,,,,,,
2026-10-15 07:00:21,POS_01,Cappuccino,43.00,NOK,CANCELLED,43.00
2026-10-15 07:01:00,POS_01,Sandwich,7.50,EUR,SUCCESS,78.00
2026-10-15 07:01:14,POS_02,Espresso,2.67,EUR,SUCCESS,27.77
2026-10-15 07:01:23,POS_01,Croissant,3.51,EUR,SUCCESS,36.50
2026-10-15 07:01:28,POS_02,Croissant,3.96,EUR,SUCCESS,41.18
...,...,...,...,...,...,...
2026-10-15 09:06:20,POS_01,Latte,47.90,NOK,CANCELLED,47.90
2026-10-15 09:06:29,POS_03,Espresso,31.40,NOK,SUCCESS,31.40
2026-10-15 09:08:08,POS_03,Latte,4.83,EUR,SUCCESS,50.23


Define a function that resamples transactions into 1-hour intervals. 

In [25]:
def resample_hourly(transactions: pd.DataFrame) -> pd.DataFrame:
    """Aggregate transaction prices into hourly intervals by item.

    We group transactions only by item and ignore all differences
    in other columns (e.g. currency, status).

    Parameters
    ----------
    transactions : pd.DataFrame
        Transaction records with a DatetimeIndex and columns for item, price.

    Returns
    -------
    pd.DataFrame
        Hourly price totals grouped by item.
    """

    return (
        transactions
        .groupby("item")
        .resample("h")
        .agg(total_price=("price", "sum"))
    )

### Task 5